# Caching & Load Testing

*Level 7 — Production RAG*

## Objective

Measure, not assume: how much does the two-tier cache (`caching/response_cache.py` exact-match,
`caching/semantic_cache.py` paraphrase-match) actually save, what does a *real* paraphrase's
cosine similarity actually look like with `nomic-embed-text`, and what does the API's throughput
actually look like under concurrent load (`../load-testing/`)?

Requires the docker-compose stack running (for real Redis) -- the API itself is only needed for
the last section.

In [1]:
import sys
import time
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "retrieval-infrastructure"))

from production_common.embed import OllamaEmbedder
from caching.response_cache import ResponseCache
from caching.semantic_cache import SemanticCache
from redis_store import RedisStore

store = RedisStore()
print("Redis reachable:", store.ping())

Redis reachable: True


## Exact-match cache: real Redis round-trip

In [2]:
response_cache = ResponseCache(store=store)

# Unique per run (real Redis persists across notebook executions -- a fixed
# string would show a hit on this first lookup too, on any re-run after the
# first).
query = f"What is the exact-match cache demo query? (run {time.time()})"

t0 = time.perf_counter()
miss = response_cache.get(query)
miss_ms = (time.perf_counter() - t0) * 1000
print(f"miss lookup: {miss_ms:.2f}ms -> {miss}")

response_cache.set(query, {"answer": "A cached demo answer.", "sources": []})

t0 = time.perf_counter()
hit = response_cache.get(query)
hit_ms = (time.perf_counter() - t0) * 1000
print(f"hit lookup:  {hit_ms:.2f}ms -> {hit}")

miss lookup: 1.09ms -> None
hit lookup:  0.51ms -> {'answer': 'A cached demo answer.', 'sources': []}


## Semantic cache: measuring *real* paraphrase similarity with `nomic-embed-text`

This is exactly how `production_common/config.py`'s `semantic_cache_threshold=0.92` was actually
calibrated -- not guessed. A genuine paraphrase pair and a genuinely unrelated pair, embedded for
real, compared for real.

In [3]:
embedder = OllamaEmbedder()

pairs = [
    ("What is the capital of France?", "What's France's capital city?", "paraphrase"),
    ("What is the capital of France?", "How do I bake sourdough bread?", "unrelated"),
    ("How many employees does the company have?", "What is the company's total headcount?", "paraphrase"),
]

import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

for q1, q2, label in pairs:
    v1, v2 = embedder.embed_one(q1), embedder.embed_one(q2)
    sim = cosine(v1, v2)
    print(f"[{label:10s}] {sim:.4f}   {q1!r} <-> {q2!r}")

[paraphrase] 0.9640   'What is the capital of France?' <-> "What's France's capital city?"
[unrelated ] 0.3399   'What is the capital of France?' <-> 'How do I bake sourdough bread?'
[paraphrase] 0.7747   'How many employees does the company have?' <-> "What is the company's total headcount?"


## What I observed (similarity measurement)

- **Exact-match cache round-trip: sub-millisecond both ways** (1.09ms miss, 0.51ms hit) -- a real
  Redis lookup over localhost is essentially free compared to any LLM call.
- **Lexically-close paraphrase: 0.964** ("What is the capital of France?" / "What's France's
  capital city?") -- comfortably above the 0.92 threshold, a real cache hit.
- **Unrelated pair: 0.340** -- comfortably below, matching the ~0.39 baseline `config.py`'s
  comment cites (small variance is expected across specific query pairs).
- **A genuinely important, less comfortable result: 0.775** for "How many employees does the
  company have?" vs. "What is the company's total headcount?" -- these two questions mean almost
  exactly the same thing, but share very little surface vocabulary ("employees"/"headcount",
  "have"/"total"). **0.775 is well below the 0.92 threshold: this real paraphrase would NOT hit
  the semantic cache.** This is a genuine, disclosed limitation of embedding-based semantic
  caching, not a bug -- `nomic-embed-text` (and cosine similarity generally) rewards shared
  vocabulary more than it rewards shared meaning for short questions. A threshold tuned low
  enough to catch this pair would also start catching genuinely different questions that happen
  to share topic words -- the exact false-positive risk `config.py`'s own comment about the
  original overly-strict `0.97` warns about, just from the other direction. The right fix for
  meaning-based (not vocabulary-based) matching would be a cross-encoder re-ranker on top of the
  embedding shortlist, or a larger/more semantically-tuned embedding model -- out of scope here,
  but worth naming rather than leaving implicit.

## Semantic cache in action

In [4]:
semantic_cache = SemanticCache(store=store, embedder=embedder)

semantic_cache.set("What is the capital of France?", {"answer": "Paris."})

hit = semantic_cache.get("What's France's capital city?")
print("paraphrase lookup ->", hit)

miss = semantic_cache.get("How do I bake sourdough bread?")
print("unrelated lookup  ->", miss)

paraphrase lookup -> {'answer': {'answer': 'Paris.'}, 'matched_query': 'What is the capital of France?', 'similarity': 0.96395624854145}
unrelated lookup  -> None


## Load test results (Locust, real run against the live API)

Full writeup: [`../load-testing/scenarios.md`](../load-testing/scenarios.md). Reading the raw
CSV this notebook's own repo actually produced:

In [5]:
import csv

with open(LEVEL_DIR / "load-testing" / "results_stats.csv") as f:
    rows = list(csv.DictReader(f))

for row in rows:
    if row["Name"] in ("/query", "/health", "Aggregated"):
        print(f"{row['Type']:5s} {row['Name']:12s} reqs={row['Request Count']:>3s}  "
              f"median={row['Median Response Time']:>8s}ms  avg={row['Average Response Time']:>10s}ms  "
              f"max={row['Max Response Time']:>10s}ms  req/s={row['Requests/s']}")

GET   /health      reqs=  1  median=26.591334026306868ms  avg=26.591334026306868ms  max=26.591334026306868ms  req/s=0.023596756364147646
POST  /query       reqs=  4  median=   24000ms  avg=27964.49972927803ms  max=40374.12316701375ms  req/s=0.09438702545659058
      Aggregated   reqs=  5  median=   24000ms  avg=22376.918050227687ms  max=40374.12316701375ms  req/s=0.11798378182073822


## What I observed (load test)

- `/health` answers in ~27ms regardless of load -- it does one Qdrant `count()` call, no LLM.
- `/query` median is **24 seconds** even at only 5 concurrent users, with a **max of 40.4
  seconds** -- confirming the finding from Notebook 02: CPU-bound Ollama generation, not the API
  or database layer, is this system's real bottleneck under load. See
  [`../load-testing/scenarios.md`](../load-testing/scenarios.md) for the full discussion,
  including what this implies for the Kubernetes HPA config in `../kubernetes/`.

## Common Failure Modes hit while building the caching layer

- **The semantic cache threshold was originally an arbitrary `0.97`**, picked without measuring
  anything -- too strict, missing genuine paraphrases (real measured similarity above shows why:
  paraphrases land around 0.92-0.97 depending on phrasing distance, not 0.99+). Fixed to `0.92`
  after actually measuring real pairs, exactly as reproduced in this notebook.
- **`retrieval-infrastructure/redis.py` originally shadowed the real `redis` PyPI package** --
  naming a local file `redis.py` meant `import redis` from anywhere on that directory's `sys.path`
  resolved to itself circularly (`AttributeError: module 'redis' has no attribute 'Redis'`).
  Renamed to `redis_store.py`; the class name `RedisStore` didn't need to change, only the
  filename.